# Neural Network Month · Project 3
## Notebook 1 of 3 — Mapping the Smell Pathway

So far this month our networks looked at numbers (Project 1) and pictures (Project 2). Today the data is a **wiring diagram**: which neuron talks to which. That is exactly what a **graph neural network (GNN)** is built for.

**The data:** a connectome of the fruit fly's brain (*Drosophila*), traced neuron by neuron from electron-microscope images. The tables we use come from a study of the fly's **smell pathway**, and they sort 5,877 neurons into five stages of the journey from nose to body:

| Class | What it does |
|---|---|
| **Receptor** (ALRN) | Sensory cells that detect odours (and temperature or humidity) |
| **Local** (ALLN) | Short-range neurons that tune signals inside the antennal lobe |
| **Projection** (ALPN) | Carry the signal from the antennal lobe into the brain |
| **Third-Order** (TOON) | Higher brain neurons that interpret the smell |
| **Descending** (DN) | Send commands down to the body |

**The task for this project:** given a neuron's measurements *and its wiring*, predict which of the five classes it belongs to. Notebook 3 asks whether knowing a neuron's neighbours helps beyond its own measurements.

**This notebook:** build the node table, get the wiring, and draw the data. Four figures.

**Credit and licence.** These tables come from Schlegel, Bates et al. (2021), *eLife*, whose supplementary data are published under CC BY 4.0; the underlying connectome is the Janelia FlyEM hemibrain (Scheffer et al. 2020), also CC BY. A Kaggle copy may label the data CC0, but the original attribution requirement still applies, so credit the authors. (This is not legal advice.)

## 1 · Setup and palette

In [ ]:
import os, sys, glob, zipfile, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch, Patch
from pathlib import Path
from sklearn.model_selection import StratifiedKFold

# One palette for the series: steel blue and lime green lead, like a lit-up brain scan.
P = {
    "steel":  "#4682B4",  # steel blue
    "lime":   "#8CC63F",  # lime green
    "teal":   "#2FB5A6",
    "violet": "#7A6BC4",
    "berry":  "#C2417E",
    "ink":    "#1E2B3A",  # text
    "paper":  "#F2F6F9",  # background
    "grid":   "#D3DCE4",
    "stone":  "#8896A5",  # neutral
    "rust":   "#B5452D",  # warnings (proxy graph)
}
# The smell pathway, nose to brain to body: lime -> teal -> steel -> violet -> berry
CLASSES = ["ALRN", "ALLN", "ALPN", "TOON", "DN"]
CLASS_NAMES = {"ALRN": "Receptor", "ALLN": "Local", "ALPN": "Projection", "TOON": "Third-Order", "DN": "Descending"}
CLASS_COLORS = {"ALRN": P["lime"], "ALLN": P["teal"], "ALPN": P["steel"], "TOON": P["violet"], "DN": P["berry"]}
MODEL_COLORS = {"Linear Regression": P["stone"], "Random Forest": P["steel"], "GCN": P["lime"]}
SEQ_CMAP = LinearSegmentedColormap.from_list("seq", [P["paper"], P["steel"], P["ink"]])

plt.rcParams.update({
    "figure.facecolor": P["paper"], "axes.facecolor": P["paper"],
    "axes.edgecolor": P["ink"], "axes.labelcolor": P["ink"],
    "text.color": P["ink"], "xtick.color": P["ink"], "ytick.color": P["ink"],
    "axes.grid": True, "grid.color": P["grid"], "grid.alpha": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "figure.titlesize": 16, "figure.titleweight": "bold",
    "font.size": 11, "figure.dpi": 110,
})

Path("figures").mkdir(exist_ok=True)
def save_fig(fig, name):
    fig.savefig(f"figures/{name}.png", dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())

def proxy_note(fig, y=-0.03):
    """Stamp figures that depend on a stand-in graph, so nobody mistakes them for real wiring."""
    if globals().get("EDGE_SOURCE", "") == "proxy":
        fig.text(0.5, y, "PROXY GRAPH: neighbours chosen by feature similarity, not real synapses",
                 ha="center", color=P["rust"], fontsize=10, fontweight="bold")

## 2 · Get the data

Download `archive.zip` from the Kaggle page, upload it to Colab (left panel → Files), and run the next cell.

In [ ]:
# Where is the data? Download archive.zip from the Kaggle page and upload it to /content
# (the Colab file panel), or point ZIP_PATH at it. The cell also tries a Kaggle download.
ZIP_PATH = "/content/archive.zip"
DATA_DIR = "fly_data"

def locate_root(base):
    for p in glob.glob(f"{base}/**/S1_hemibrain_neuron_layers.csv", recursive=True):
        return os.path.dirname(p)
    return None

root = locate_root(DATA_DIR) if os.path.isdir(DATA_DIR) else None
if root is None and os.path.exists(ZIP_PATH):
    zipfile.ZipFile(ZIP_PATH).extractall(DATA_DIR)
    root = locate_root(DATA_DIR)
if root is None:
    try:
        import kagglehub
        root = locate_root(kagglehub.dataset_download("thedevastator/hemibrain-neuronal-connectome"))
    except Exception as e:
        print("Kaggle download did not work:", e)
if root is None:
    raise FileNotFoundError("Could not find the hemibrain CSVs. Upload the Kaggle zip and set ZIP_PATH.")
print("Data folder:", root)

## 3 · The node table, and the only cleaning we do

Each neuron becomes one row. We keep five **measured properties**: `pre` (output synapse sites), `post` (input synapse sites), `downstream` (total outgoing connections), `voxels` (physical size) and `soma` (whether the cell body is present). We deliberately leave out anything that names or describes the class (cell type, name, layer), because that would hand the model the answer.

The cleaning steps, and why:

1. **Set aside neurons listed in two class tables.** They have no single true label.
2. **Drop `total.length` and `upstream`.** `upstream` is identical to `post` in every row, and `total.length` is missing for a handful of neurons, so leaving it out avoids guessing values.
3. **Keep orphan fragments.** A few dozen receptor neurons are unattached pieces; their labels are still valid.

In [ ]:
# ---- The only places we touch the data ----
N_FOLDS, SEED = 5, 42
FEATURES = ["pre", "post", "downstream", "voxels", "soma"]   # measured properties, no label information

def build_nodes(root):
    files = {"ALRN": "S3_hemibrain_ALRN_meta.csv", "ALLN": "S4_hemibrain_ALLN_meta.csv",
             "ALPN": "S5_hemibrain_ALPN_meta.csv", "TOON": "S6_hemibrain_TOON_meta.csv",
             "DN": "S7_hemibrain_DN_meta.csv"}
    cols = ["bodyid", "status"] + FEATURES
    nodes = pd.concat([pd.read_csv(os.path.join(root, f), usecols=cols).assign(cls=c) for c, f in files.items()],
                      ignore_index=True)
    # A neuron listed in two class tables has no single ground-truth label, so it is set aside.
    ambiguous = nodes.bodyid[nodes.bodyid.duplicated(keep=False)].unique()
    nodes = nodes[~nodes.bodyid.isin(ambiguous)].reset_index(drop=True)
    nodes["soma"] = nodes["soma"].astype(int)
    nodes["label"] = nodes.cls.map(CLASSES.index)
    # The authors' layer score is only used to DRAW the pathway, never as a feature or a target.
    layers = pd.read_csv(os.path.join(root, "S1_hemibrain_neuron_layers.csv")).rename(columns={"bodyId": "bodyid"})
    nodes = nodes.merge(layers[["bodyid", "layer_mean"]], on="bodyid", how="left")
    nodes["fold"] = -1
    for k, (_, te) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(nodes, nodes.label)):
        nodes.loc[te, "fold"] = k
    return nodes, len(ambiguous)

In [ ]:
nodes, n_ambiguous = build_nodes(root)
print(f"Neurons listed in two class tables and set aside: {n_ambiguous}")
print(f"Neurons kept: {len(nodes)} | orphan fragments kept: {(nodes.status == 'Orphan').sum()} | missing feature values: {int(nodes[FEATURES].isna().sum().sum())}")
nodes.groupby("cls").size().reindex(CLASSES).rename(CLASS_NAMES)

## 4 · Figure 1 — the five classes

In [ ]:
counts = nodes.cls.value_counts().reindex(CLASSES)
fig, ax = plt.subplots(figsize=(9, 4.6))
bars = ax.bar(range(5), counts.values, color=[CLASS_COLORS[c] for c in CLASSES], width=0.7)
ax.bar_label(bars, labels=[f"{v:,}\n{v/counts.sum():.0%}" for v in counts.values], padding=3, fontsize=10)
ax.set_xticks(range(5)); ax.set_xticklabels([CLASS_NAMES[c] for c in CLASSES])
ax.set_ylabel("Neurons"); ax.set_ylim(0, counts.max() * 1.18)
ax.set_title("Neurons by Class")
fig.tight_layout(); save_fig(fig, "01_neurons_by_class"); plt.show()

Two classes (receptor and third-order) make up about 85% of the neurons, while *Local* has under 200. That imbalance matters: a model can score high **accuracy** just by favouring the big classes, which is why Notebook 3 also reports macro-averaged F1, precision and recall, where every class counts equally.

## 5 · Figure 2 — the signal flows downhill

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
for i, c in enumerate(CLASSES):
    v = nodes.loc[nodes.cls == c, "layer_mean"].dropna().values
    parts = ax.violinplot(v, positions=[i], widths=0.8, showextrema=False)
    for b in parts["bodies"]:
        b.set_facecolor(CLASS_COLORS[c]); b.set_edgecolor(P["ink"]); b.set_alpha(0.85)
    ax.hlines(np.median(v), i - 0.3, i + 0.3, color=P["ink"], linewidth=2.2)
ax.set_xticks(range(5)); ax.set_xticklabels([CLASS_NAMES[c] for c in CLASSES])
ax.set_ylabel("Layer score (nose to body)")
ax.set_title("Smell Signal Flow")
fig.tight_layout(); save_fig(fig, "02_smell_signal_flow"); plt.show()

This is the pathway. Receptor neurons sit at the start, then local and projection neurons, then the higher brain, then descending neurons at the far end. The **layer score** comes from the original authors, who computed it by tracing signal flow through the connectome. We use it only to *draw* the pathway here, never as a feature or a target, because it was derived from the wiring itself.

## 6 · The wiring

A GNN needs **edges**: who connects to whom. This dataset has none, so we get them from the **neuPrint** server at Janelia, which hosts the full hemibrain connectome.

**To use the real wiring (recommended):**
1. Log in at neuprint.janelia.org with a Google account and copy your **auth token** from your account menu (the exact menu wording may differ).
2. In Colab, click the key icon (Secrets), add a secret named `NEUPRINT_TOKEN`, paste the token, and switch on notebook access. Or just paste it when the cell asks.

The cell asks the server for every connection **between our neurons** with at least `MIN_SYNAPSES` synapses, and it first checks which data version matches our neuron IDs best, since the supplement and the server can differ slightly.

**If there is no token or the server fails**, the notebook falls back to a **proxy graph**: each neuron is linked to its ten most similar neurons by measured features. That is *not* wiring, so every figure that depends on it is stamped with a red warning. Use it to test the pipeline, not to report findings.

In [ ]:
# ---- Where the wiring comes from ----
# Main route: the neuPrint server at Janelia (needs a free account token).
# Backup route: a clearly labelled PROXY graph built from the node table alone.
MIN_SYNAPSES      = 3                    # ignore connections weaker than this (a common guard against false synapse calls)
NEUPRINT_SERVER   = "neuprint.janelia.org"
NEUPRINT_DATASETS = ["hemibrain:v1.2.1", "hemibrain:v1.2", "hemibrain:v1.1"]   # the one that matches our IDs best is used
EDGES_CSV         = None                 # optional: path to your own edge file with columns bodyId_pre, bodyId_post, weight
USE_PROXY_IF_NEEDED = True

def get_token():
    """Looks for NEUPRINT_TOKEN in environment variables or Colab Secrets; in Colab, offers a hidden prompt."""
    tok = os.environ.get("NEUPRINT_TOKEN")
    if tok: return tok
    try:
        from google.colab import userdata
        tok = userdata.get("NEUPRINT_TOKEN")
        if tok: return tok
    except Exception:
        pass
    if "google.colab" in sys.modules:
        import getpass
        return getpass.getpass("neuPrint token (press Enter to skip and use the proxy graph): ").strip() or None
    return None

def fetch_edges_neuprint(ids, token):
    try:
        import neuprint
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q neuprint-python")
        import neuprint
    from neuprint import Client, fetch_adjacencies
    ids = [int(i) for i in ids]
    sample = [int(i) for i in np.random.default_rng(0).choice(ids, size=min(1000, len(ids)), replace=False)]
    best = None
    for ds in NEUPRINT_DATASETS:                      # the supplement and the server can differ by a data version
        try:
            client = Client(NEUPRINT_SERVER, dataset=ds, token=token)
            found = int(client.fetch_custom(f"MATCH (n:Neuron) WHERE n.bodyId IN {sample} RETURN count(n) AS n")["n"].iloc[0])
        except Exception as e:
            print(f"  {ds}: could not be queried ({str(e)[:90]})"); continue
        print(f"  {ds}: {found} of {len(sample)} sampled neuron IDs found")
        if best is None or found > best[1]: best = (ds, found, client)
    if best is None: raise RuntimeError("no neuPrint dataset could be queried")
    ds, found, client = best
    print(f"Using {ds}. Fetching connections among {len(ids)} neurons (this can take a few minutes)...")
    _, conn = fetch_adjacencies(sources=ids, targets=ids, min_total_weight=MIN_SYNAPSES,
                                omit_rois=True, weight_props=["weight"], client=client)
    return conn[["bodyId_pre", "bodyId_post", "weight"]], f"neuPrint {ds}"

def proxy_edges(nodes, k=10):
    """Backup graph: link each neuron to its k most similar neurons by measured features. NOT synapses."""
    from sklearn.neighbors import NearestNeighbors
    X = np.log1p(nodes[["pre", "post", "downstream", "voxels"]].astype(float))
    X["soma"] = nodes["soma"].astype(float)
    X = ((X - X.mean()) / X.std()).values
    nn = NearestNeighbors(n_neighbors=k + 1, metric="cosine").fit(X)
    _, idx = nn.kneighbors(X)
    ids = nodes.bodyid.values
    rows = [(ids[i], ids[j], 1.0) for i in range(len(ids)) for j in idx[i, 1:]]
    return pd.DataFrame(rows, columns=["bodyId_pre", "bodyId_post", "weight"])

def get_edges(nodes):
    edges, source = None, None
    if EDGES_CSV:
        edges, source = pd.read_csv(EDGES_CSV), f"file {EDGES_CSV}"
    else:
        token = get_token()
        if token:
            try:
                edges, source = fetch_edges_neuprint(nodes.bodyid.values, token)
            except Exception as e:
                print("neuPrint route failed:", repr(e)[:300])
        else:
            print("No neuPrint token found.")
    if edges is None:
        if not USE_PROXY_IF_NEEDED: raise RuntimeError("No edges available.")
        print("\n*** Falling back to the PROXY graph: neighbours are chosen by feature similarity, NOT real synapses. ***\n")
        return proxy_edges(nodes), "proxy"
    # Necessary tidying: keep our neurons only, drop self-connections, merge repeated pairs, apply the weight floor
    keep = set(nodes.bodyid)
    edges = edges[edges.bodyId_pre.isin(keep) & edges.bodyId_post.isin(keep) & (edges.bodyId_pre != edges.bodyId_post)]
    edges = edges.groupby(["bodyId_pre", "bodyId_post"], as_index=False)["weight"].sum()
    return edges[edges.weight >= MIN_SYNAPSES].reset_index(drop=True), source

In [ ]:
edges, EDGE_SOURCE = get_edges(nodes)
deg = pd.concat([edges.bodyId_pre, edges.bodyId_post]).value_counts()
print(f"Edge source: {EDGE_SOURCE}")
print(f"{len(edges):,} connections among {len(nodes):,} neurons | median connections per neuron: {deg.reindex(nodes.bodyid).fillna(0).median():.0f} | "
      f"neurons with none: {(deg.reindex(nodes.bodyid).fillna(0) == 0).sum()}")

## 7 · Figure 3 — who connects to whom

In [ ]:
cls_of = dict(zip(nodes.bodyid, nodes.cls))
e = edges.assign(pre_c=edges.bodyId_pre.map(cls_of), post_c=edges.bodyId_post.map(cls_of))
M = e.pivot_table(index="pre_c", columns="post_c", values="weight", aggfunc="sum", fill_value=0).reindex(index=CLASSES, columns=CLASSES, fill_value=0)
Mn = M.div(M.sum(1).replace(0, np.nan), axis=0).fillna(0)

fig, ax = plt.subplots(figsize=(7.6, 6.2))
ax.imshow(Mn.values, cmap=SEQ_CMAP, vmin=0, vmax=max(Mn.values.max(), 1e-9)); ax.grid(False)
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{Mn.values[i, j]:.0%}", ha="center", va="center", fontsize=12, fontweight="bold",
                color="white" if Mn.values[i, j] > 0.4 * Mn.values.max() else P["ink"])
names = [CLASS_NAMES[c] for c in CLASSES]
ax.set_xticks(range(5)); ax.set_xticklabels(names, rotation=20); ax.set_yticks(range(5)); ax.set_yticklabels(names)
ax.set_xlabel("Receiving neuron class"); ax.set_ylabel("Sending neuron class")
ax.set_title("Who Connects to Whom")
fig.tight_layout(); proxy_note(fig, y=-0.02); save_fig(fig, "03_who_connects_to_whom"); plt.show()

Each row shows where a class sends its output (rows add to 100%). With real wiring, the pattern should follow the pathway: signals mostly move *forward* through the stages. That is the reason graphs help. **A neuron's partners tell you what stage it belongs to**, information that a table of its own measurements does not contain.

## 8 · Figure 4 — two hops of context

In [ ]:
import networkx as nx
from collections import defaultdict
nbrs = defaultdict(dict)
for a, b, w in zip(edges.bodyId_pre, edges.bodyId_post, edges.weight):
    nbrs[a][b] = nbrs[a].get(b, 0) + w
    nbrs[b][a] = nbrs[b].get(a, 0) + w
alpn = nodes.bodyid[nodes.cls == "ALPN"]
alpn_deg = pd.Series({i: len(nbrs.get(i, {})) for i in alpn})
center = (alpn_deg - alpn_deg.median()).abs().idxmin()          # a typical projection neuron
hop1 = sorted(nbrs[center], key=lambda b: -nbrs[center][b])[:10]
hop2 = []
for h in hop1:
    hop2 += [b for b in sorted(nbrs[h], key=lambda b: -nbrs[h][b]) if b != center and b not in hop1 and b not in hop2][:3]
members = [center] + hop1 + hop2
G = nx.Graph()
G.add_nodes_from(members)
for a in members:
    for b, w in nbrs.get(a, {}).items():
        if b in members and a < b: G.add_edge(a, b, weight=w)
pos = nx.shell_layout(G, nlist=[[center], hop1, hop2])

fig, ax = plt.subplots(figsize=(8.6, 8.2)); ax.axis("off"); ax.grid(False)
nx.draw_networkx_edges(G, pos, ax=ax, edge_color=P["stone"], alpha=0.45,
                       width=[0.6 + 0.35 * np.log1p(d["weight"]) for _, _, d in G.edges(data=True)])
sizes = {center: 1500, **{h: 650 for h in hop1}, **{h: 260 for h in hop2}}
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=[CLASS_COLORS[cls_of[n]] for n in G.nodes],
                       node_size=[sizes[n] for n in G.nodes], edgecolors=P["ink"], linewidths=1.4)
ax.legend(handles=[Patch(color=CLASS_COLORS[c], label=CLASS_NAMES[c]) for c in CLASSES], frameon=False,
          loc="upper center", bbox_to_anchor=(0.5, -0.01), ncol=5, fontsize=10)
ax.set_title("Two Hops of Context")
fig.tight_layout(); proxy_note(fig, y=0.015); save_fig(fig, "04_two_hops_of_context"); plt.show()
print("Centre: one typical projection neuron. Inner ring: its strongest partners. Outer ring: their partners.")

The large node is a typical projection neuron. The ring around it holds its strongest partners, and the outer ring holds *their* partners. A two-layer GCN does exactly this: each neuron gathers information from its partners, then from the partners' partners. The colours around a neuron are clues about what it is.

**Save the graph for Notebooks 2 and 3.**

In [ ]:
nodes.to_csv("fly_nodes.csv", index=False); edges.to_csv("fly_edges.csv", index=False)
open("fly_edge_source.txt", "w").write(EDGE_SOURCE)
print("Saved fly_nodes.csv, fly_edges.csv and fly_edge_source.txt")

## What we did, and what we did not

**Changed:** set aside 16 neurons with two labels; dropped two redundant or incomplete columns; kept only connections between our own neurons with at least `MIN_SYNAPSES` synapses.
**Not changed:** no neurons removed for being fragmentary, no values imputed, no hand-picked examples.

**Limits to state in the newsletter**

- The class labels come from the original study's annotations, and some were themselves defined using connectivity. A model that "discovers" them from wiring is partly re-finding how they were defined.
- This is one fly's right half-brain; the hemibrain is a large piece, not the whole brain.
- The weight floor (`MIN_SYNAPSES`) is a modelling choice. Change it and the graph changes.

**Next:** Notebook 2 builds the graph neural network and shows how it passes messages.

In [ ]:
# Optional: download this notebook's figures
try:
    import shutil
    from google.colab import files
    shutil.make_archive("nb1_figures", "zip", "figures")
    files.download("nb1_figures.zip")
except ImportError:
    print("Figures are saved in the 'figures' folder.")